---
sidebar_label: SingleStore
---

# SingleStoreLoader

The `SingleStoreLoader` allows you to load documents directly from a SingleStore database table. It is part of the `langchain-singlestore` integration package.

## Overview

### Integration Details

| Class | Package | JS Support |
| :--- | :--- | :---: |
| `SingleStoreLoader` | `langchain_singlestore` | ❌ |

### Features
- Load documents lazily to handle large datasets efficiently.
- Supports native asynchronous operations.
- Easily configurable to work with different database schemas.

## Setup

To use the `SingleStoreLoader`, you need to install the `langchain-singlestore` package. Follow the installation instructions below.

### Installation

Install **langchain_singlestore**.

In [ ]:
%pip install -qU langchain_singlestore

## Initialization

To initialize `SingleStoreLoader`, you need to provide connection parameters for the SingleStore database and specify the table and fields to load documents from.

### Required Parameters:
- **host** (`str`): Hostname, IP address, or URL for the database.
- **table_name** (`str`): Name of the table to query. Defaults to `embeddings`.
- **content_field** (`str`): Field containing document content. Defaults to `content`.
- **metadata_field** (`str`): Field containing document metadata. Defaults to `metadata`.

### Optional Parameters:
- **id_field** (`str`): Field containing document IDs. Defaults to `id`.

### Connection Pool Parameters:
- **pool_size** (`int`): Number of active connections in the pool. Defaults to `5`.
- **max_overflow** (`int`): Maximum connections beyond `pool_size`. Defaults to `10`.
- **timeout** (`float`): Connection timeout in seconds. Defaults to `30`.

### Additional Options:
- **pure_python** (`bool`): Enables pure Python mode.
- **local_infile** (`bool`): Allows local file uploads.
- **charset** (`str`): Character set for string values.
- **ssl_key**, **ssl_cert**, **ssl_ca** (`str`): Paths to SSL files.
- **ssl_disabled** (`bool`): Disables SSL.
- **ssl_verify_cert** (`bool`): Verifies server's certificate.
- **ssl_verify_identity** (`bool`): Verifies server's identity.
- **autocommit** (`bool`): Enables autocommits.
- **results_type** (`str`): Structure of query results (e.g., `tuples`, `dicts`).

In [ ]:
from langchain_singlestore.document_loaders import SingleStoreLoader

loader = SingleStoreLoader(
    host="127.0.0.1:3306/db",
    table_name="documents",
    content_field="content",
    metadata_field="metadata",
    id_field="id",
)

## Connection options

`SingleStoreLoader` supports several ways to connect:

| Mode | When to use | Kwargs |
| --- | --- | --- |
| Direct connection kwargs | Simplest — pass `host`, `user`, `password`, `port`, `database`. | `host=...`, `user=...`, ... |
| Environment variables | Deployment / notebooks where credentials come from the environment. | Set `SINGLESTOREDB_URL` (or `SINGLESTOREDB_HOST` / `_PORT` / `_USER` / `_PASSWORD` / `_DATABASE`) and omit the kwargs. |
| Existing `singlestoredb.Connection` | Share a caller-owned connection; the loader never closes it. | `connection=my_conn` |
| Existing `sqlalchemy.pool.Pool` | Share a pre-built pool across multiple components. | `connection_pool=my_pool` |

`connection` and `connection_pool` are mutually exclusive. `pool_size`, `max_overflow`, and `timeout` are ignored when either is supplied. See the [`singlestoredb.connect` reference](https://singlestoredb-python.labs.singlestore.com/generated/singlestoredb.connect.html) for the full list of connection kwargs.


In [ ]:
import os

import singlestoredb

from langchain_singlestore.document_loaders import SingleStoreLoader
from singlestore_langchain_core import create_connection_pool

# 1) Direct connection kwargs
loader = SingleStoreLoader(host="127.0.0.1:3306/db", table_name="documents")

# 2) SINGLESTOREDB_URL environment variable — no connection kwargs needed
os.environ["SINGLESTOREDB_URL"] = "root:pass@localhost:3306/db"
loader = SingleStoreLoader(table_name="documents")

# 3) Reuse an existing singlestoredb.Connection
#    (SingleStoreLoader never closes a caller-owned connection.)
conn = singlestoredb.connect("root:pass@localhost:3306/db")
loader = SingleStoreLoader(connection=conn, table_name="documents")

# 4) Reuse a caller-managed connection pool — shareable across components
pool = create_connection_pool(
    pool_size=5,
    max_overflow=10,
    timeout=30,
    connection_kwargs={
        "host": "localhost",
        "user": "root",
        "password": "pass",
        "database": "db",
    },
)
loader = SingleStoreLoader(connection_pool=pool, table_name="documents")


## Load

In [ ]:
docs = loader.load()
docs[0]

In [ ]:
print(docs[0].metadata)

## Lazy Load

In [ ]:
page = []
for doc in loader.lazy_load():
    page.append(doc)
    if len(page) >= 10:
        # do some paged operation, e.g.
        # index.upsert(page)

        page = []